In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_parquet('../data/res/int.parquet', columns=['rating','user_id', 'parent_asin', 'date'])
df

,rating,user_id,parent_asin,date
0,5.0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,2020-05-05 14:08:48.923
1,4.0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,2020-05-04 18:10:55.070
2,5.0,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,2020-05-16 21:41:06.052
3,1.0,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,2022-01-28 18:13:50.220
4,5.0,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,2020-12-30 10:02:43.534
...,...,...,...,...
74063920,5.0,AEEDOIMWOYLWHCJUAGTTB3MPSBHQ,B08YY7J71M,2020-12-08 14:45:17.808
74063921,5.0,AEEDOIMWOYLWHCJUAGTTB3MPSBHQ,B07PDHSLM6,2020-02-06 04:18:09.889
74063922,5.0,AEEDOIMWOYLWHCJUAGTTB3MPSBHQ,B09MS2RFM1,2020-02-06 04:15:33.100
74063923,5.0,AEEDOIMWOYLWHCJUAGTTB3MPSBHQ,B013W95MWG,2017-10-25 23:06:45.577


In [3]:
df = df.sort_values('date').reset_index(drop=True)
df['is_positive'] = (df['rating']>=4).astype('int8')

df["is_positive"].value_counts(normalize=True)

is_positive
1    0.797178
0    0.202822
Name: proportion, dtype: float64

позитивных оценок(>=4) в 4 раза больше чем непозитивных(<4) 

In [4]:
train = df[(df['date'] < np.datetime64('2023-07-14'))].copy()

val = df[
        (df['date'] >= np.datetime64('2023-07-14')) & 
        (df['date'] < np.datetime64('2023-08-14'))
         ].copy()

test = df[df['date'] >= np.datetime64('2023-08-14')].copy()

разделение данных сделал так чтобы тестовой части было все взаимодействия кроме последних 2 месяцев. предпоследний месяц идет для валидации. а последний для финального теста. предсказание на месяц вперед обусловлено тем что при длительном во времени предсказании качество предсказания падает. 

In [5]:
val_pos = val[val['is_positive']==1].copy()
train_users = set(train['user_id'].unique())

warm_val = val_pos[
    val_pos["user_id"].isin(train_users)
].copy()

cold_val = val_pos[
    ~val_pos["user_id"].isin(train_users)
].copy()

In [6]:
print("Positive val users:", val_pos["user_id"].nunique())
print("Warm users:", warm_val["user_id"].nunique())
print("Cold users:", cold_val["user_id"].nunique())

print(
    "Warm share:",
    warm_val["user_id"].nunique()
    / val_pos["user_id"].nunique()
)

Positive val users: 47497
Warm users: 13620
Cold users: 33877
Warm share: 0.2867549529443965


холодных пользователей 72% от всех пользователей котрые попали в валидационную выборку

In [7]:
train_seen = train[['parent_asin','user_id']].drop_duplicates()

In [8]:
warm_target = warm_val.merge(
    train_seen.assign(seen=1),
    on=["parent_asin", "user_id"],
    how="left"
)

warm_target = warm_target[warm_target['seen'].isna()].drop(columns='seen')
warm_user_targets = warm_target.groupby('user_id')['parent_asin'].agg(set)

cold_user_targets = (
    cold_val
    .groupby("user_id")["parent_asin"]
    .agg(set)
)

In [9]:
print("Warm target users:", len(warm_user_targets))
print("Cold target users:", len(cold_user_targets))

Warm target users: 13615
Cold target users: 33877


известных пользователей стало меньше тк 5 человек оставили отзыв на тех родительских айди товаров к которым они уже делали отзыв. либо изменили 

In [10]:
warm_user_targets

user_id
AE22JHXUEFSEC4P4GITEB46DLI7A                            {B0CC3VSXK7}
AE22NKANL27CSAIG64QEDL2TBF4A                            {B00UKUHWOM}
AE23GZBUT7UPKGYFI4E37J5XMQDA                            {B0BML57NSW}
AE23SZMIL4VUVEKWURCYB2ZQV46Q                            {B0BXP4899F}
AE252QUUKZ3QEHC46FGTG27YLMPQ                            {1335457615}
                                                ...                 
AHZYREJJXDW4QSL33G5KPDQHQ3VQ    {B0BNJV82CH, 1250891906, B0C2S9T76R}
AHZYWFX5UZUSKBMWAKJYSPQ6RS6Q                            {1728256151}
AHZZDAHRB3OYCLO2KBBG27KVXTEQ                {1773066641, 1797211757}
AHZZRVH6XL4BJ5RRUFJ3QGN3OYCA                {B0CCYKPSTL, B081YNQ9RV}
AHZZUC6PRHSTZWUXWWW763D4OCAA                            {B0CFZCJ3H8}
Name: parent_asin, Length: 13615, dtype: object

In [11]:
train_user_hist = train.groupby('user_id')['parent_asin'].agg(set)

In [12]:
def recall_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans) 

    if len(true_ans) ==0:
        return 0
    
    hits = len(set(recom) & true_ans)

    return hits / len(true_ans)

In [13]:
def hit_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    return int(
        len(set(recom) & true_ans) > 0
    )

в качестве проверки качества выбраны метрики recall@k и hitrate@k. Тк задача состоит в том чтобы система выдала наиболее релевантные рекомендации и правильно их отранжировала. а чтобы ранжировщик мог ранжировать сначала генераторы должны хотя бы уметь находить кандидатов. и recall@k и hitrate@k как раз показывают сколько из всего таргетов мы нашли и для скольких пользователей мы нашли хотябы 1 таргет

In [14]:
train_positive = train[train['is_positive']==1].copy()
item_popularity = train_positive.groupby('parent_asin').size().sort_values(ascending=False)

item_popularity.head(20)

parent_asin
B075X8471B    148503
B07GZFM1ZM    121752
B01K8B8YA8     97276
B07H65KP63     85615
B010BWYDYA     79544
B0791TX5P5     75243
B08XPWDSWW     57507
B07KTYJ769     45649
B07S764D9V     44377
B0BW4PFM58     42416
B07456BG8N     38782
B07HZLHPKP     38410
B07PHQ93TV     35945
B08XNCHTCY     34778
B08F1P3BCC     34161
B00OQVZDJM     33494
B00L9B7IKE     31647
B01MTF2Z37     29414
B07VXXBTX4     29405
B07N8VFFNS     29062
dtype: int64

в качестве бейзлайна выбрал рекомендации по популярным товаров тк это вариант наиболее простой и позволяет давать рекомендации холодным пользователем

In [15]:
popular_items = item_popularity.index.tolist()


In [16]:
def recommend_popular(user_id, popular_items, user_history, k):

    seen = user_history.get(user_id, set())

    recommendations = []

    for item in popular_items:
        if item not in seen:
            recommendations.append(item)

        if len(recommendations) == k:
            break

    return recommendations

In [17]:
recall = []
Hit = []


for user_id, targets in warm_user_targets.items():

    recs = recommend_popular(user_id, popular_items, train_user_hist, 1000)

    recall_k = recall_at_k(recs, targets, k=1000)
    hit_k = hit_at_k(recs, targets, k=1000)

    recall.append(recall_k)
    Hit.append(hit_k)

print('recall@1000:', np.mean(recall))
print('hitRates@1000:', np.mean(Hit))

recall@1000: 0.04668191277445777
hitRates@1000: 0.055526992287917736


метрики качества для baseline ожидаемые. использование популярных товаров за все время не отражают предпочтения пользователей. особенно в екомерции где у каждого человека предпочтения меняются от сессии к сессии. также товары которые живут дольше могут иметь больше отзывов относительно тех что появились недавно. а тк мы брали отзывы за все время у нас много старых товаров. поэтому дальше сделаю тоже самое но с окном не во всю историю а за определенные дни до валидации

In [18]:
def pop_k_days(k, train):
    end = train['date'].max()
    train_k_days = train[train['date']>=(end - pd.Timedelta(days=k))].copy()

    pos_train = train_k_days[train_k_days['is_positive'] == 1]
    popular_it = pos_train.groupby('parent_asin').size().sort_values(ascending=False)

    popular_items = popular_it.index.tolist()

    return popular_items

In [19]:
def rec_k_days(user_id, user_hist, popular_items, at_K):

    seen = user_hist.get(user_id, set())
    rec = []

    for i in popular_items:
        if i not in seen:
            rec.append(i)

        if len(rec) == at_K:
            break
    
    return rec 
        

In [20]:
def scores(list_of_days, list_of_k, train, user_hist, val_user_targets):

    results = []

    for i in list_of_days:
        popular_items = pop_k_days(i, train)
        
        for k in list_of_k:

            recall_scores = []
            hit_scores = []

            for user_id, targets in val_user_targets.items():

                recs = rec_k_days(user_id, user_hist, popular_items, k)

                recall_score = recall_at_k(recs, targets, k)
                hit_score = hit_at_k(recs, targets, k)

                recall_scores.append(recall_score)
                hit_scores.append(hit_score)

            results.append({
                "days": i,
                "k": k,
                "recall": np.mean(recall_scores),
                "hit_rate": np.mean(hit_scores)
            })

    return pd.DataFrame(results)

In [21]:
days = [360, 180, 90, 30]
k = [10, 50, 100, 300, 1000]

res = scores(days, k, train, train_user_hist, warm_user_targets)
res

,days,k,recall,hit_rate
0,360,10,0.007466,0.008667
1,360,50,0.024045,0.027470
2,360,100,0.038599,0.044142
3,360,300,0.067724,0.076460
4,360,1000,0.127402,0.143665
5,180,10,0.006089,0.006904
6,180,50,0.026500,0.030040
7,180,100,0.040102,0.044583
8,180,300,0.073541,0.082850
9,180,1000,0.140689,0.158281


чем меньше окно и чем больше k тем выше качетсво. самые высокие значения на окне в 30 дней и k в 1000. это логично тк мы фильтруем старые товары и также при высоком k у нас увеличивается количество товаров и шанс угадать с верным. я бы взял для популярных товаров k=1000 days = 90. тк разницы большой в метриках нет. но данные за 90 дней более стабильные

In [ ]:
cold_res = scores(days, k, train, train_user_hist, cold_user_targets)
cold_res

,days,k,recall,hit_rate
0,360,10,0.008825,0.011158
1,360,50,0.032665,0.039791
2,360,100,0.049699,0.059893
3,360,300,0.084541,0.101072
4,360,1000,0.170568,0.197479
5,180,10,0.008161,0.009977
6,180,50,0.034658,0.041326
7,180,100,0.051020,0.059982
8,180,300,0.092157,0.108628
9,180,1000,0.185032,0.212858


: 

как можно увидить для холодных пользователей которые за последние время не писали никакие отзывы рекомендации по популярности немного лучще чем для теплных